# 04 - Rolling economy as an absorbing Markov chain
**Question:** how much gold does it cost to reach a 2-star (3 copies) or 3-star (9 copies) of a unit, given my level, the unit's cost tier, and how many other players are fighting for it?

**Model.** The state is the number of copies of the target unit I hold. Each shop view shows 5 slots. A slot shows the target with probability

`p = (chance the slot is this cost tier at my level) x (copies of the target left) / (copies left in that tier)`

and the number of hits in one shop is Binomial(5, p). Hits move me up the state ladder, and reaching the needed number of copies is the absorbing state. The fundamental matrix gives the expected number of shops, a Monte Carlo run gives the spread.

The probability depends on the state, because each copy I buy leaves the pool. That is what makes it a Markov chain and not a simple geometric wait.

## 1. Game configuration (Set 18)
Sources: the SeeMeta Set 18 odds page (levels 1 and 11) and a shop-odds table supplied by the project owner (levels 2-10). They agree everywhere except level 7, where this notebook uses 16/30/43/10/1. **Spot-check against the in-game odds before trusting the results.**
Odds and pool sizes can change between sets and patches.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from math import comb
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# shop odds in percent: rows = player level, columns = unit cost
# Level 7 follows the in-game style table (16/30/43/10/1). SeeMeta lists 19/30/40/10/1 for level 7, so the two sources disagree.
# Levels 2-6 and 8-10 agree between them. Level 1 and level 11 come from SeeMeta only. Check against your own client.
ODDS = pd.DataFrame(
    [[100, 0, 0, 0, 0], [100, 0, 0, 0, 0], [75, 25, 0, 0, 0], [55, 30, 15, 0, 0],
     [45, 33, 20, 2, 0], [30, 40, 25, 5, 0], [16, 30, 43, 10, 1], [15, 20, 32, 30, 3],
     [10, 17, 25, 33, 15], [5, 10, 20, 40, 25], [1, 2, 12, 50, 35]],
    index=range(1, 12), columns=range(1, 6)) / 100
assert np.allclose(ODDS.sum(axis=1), 1), "each level's odds must sum to 100%"

# pool: number of different champions per cost, and copies of each
POOL = pd.DataFrame({"champions": [14, 13, 14, 14, 10],
                     "copies_each": [30, 25, 18, 10, 9]}, index=range(1, 6))
POOL["tier_total"] = POOL["champions"] * POOL["copies_each"]

SLOTS = 5            # shop slots per refresh
REROLL_COST = 2      # gold per refresh
OTHERS = 7           # other players in the lobby
COPIES_FOR_STAR = {2: 3, 3: 9}
POOL

## 2. Estimate pool pressure from your own match data
Two numbers come from `units.csv`: how many copies a player usually holds of a unit they field, and how much of each cost tier an average board occupies (times 7 other players = background depletion of the pool).

Assumption to verify: `rarity + 1` equals the unit's gold cost. The first table checks it: the number of distinct units seen per rarity should be at or just below the champions per cost in the pool (14, 13, 14, 14, 10).

These are end-of-game boards, so the depletion estimate is an upper-ish bound for early and mid game.

In [ ]:
DATA = next(p for p in [Path("data"), Path("../data"), Path("D:/VS Code/data")] if (p / "units.csv").exists())
units = pd.read_csv(DATA / "units.csv")
units["unit"] = (units["unit"].str.replace(r"DA_(18_)?", "", regex=True)
                              .str.replace(r"18(?=[_:|]|$)", "", regex=True))
units["cost"] = units["rarity"] + 1
units["copies"] = units["tier"].map({1: 1, 2: 3, 3: 9}).fillna(9)

seen = units.groupby("cost")["unit"].nunique()
print(pd.DataFrame({"units_seen": seen, "champions_in_pool": POOL["champions"]}))

n_boards = units["board_id"].nunique()
held_when_played = units.groupby("cost")["copies"].mean()
per_board = units.groupby("cost")["copies"].sum() / n_boards
BG = (OTHERS * per_board / POOL["tier_total"]).clip(upper=0.9).fillna(0)

pd.DataFrame({"copies_held_per_fielded_unit": held_when_played.round(2),
              "tier_copies_per_board": per_board.round(1),
              "background_depletion": BG.round(3)})

## 3. The chain

In [ ]:
def slot_probs(level, cost, n_need, contesters=0, bg_scale=1.0):
    '''Per-slot chance of the target unit for state s = copies I hold, s = 0 .. n_need-1.
    Returns None if the pool can never supply n_need copies.'''
    total = POOL.loc[cost, "copies_each"]
    taken_by_others = contesters * float(held_when_played.get(cost, 3.0))
    if total - taken_by_others < n_need:
        return None
    other_left = (POOL.loc[cost, "tier_total"] - total) * (1 - bg_scale * float(BG.get(cost, 0.0)))
    s = np.arange(n_need)
    target_left = total - taken_by_others - s
    p = ODDS.loc[level, cost] * target_left / (target_left + other_left)
    return np.clip(p, 0, 1)


def transition_Q(pr):
    '''Transient part of the transition matrix: from state i, hits ~ Binomial(5, p_i); absorbing state dropped.'''
    m = len(pr)
    Q = np.zeros((m, m))
    for i in range(m):
        for h in range(SLOTS + 1):
            j = i + h
            if j < m:
                Q[i, j] += comb(SLOTS, h) * pr[i] ** h * (1 - pr[i]) ** (SLOTS - h)
    return Q


def expected_gold(level, cost, n_need, owned=0, contesters=0, bg_scale=1.0):
    '''Expected gold from `owned` copies up to n_need copies. The first shop view is free, every refresh costs 2.'''
    pr = slot_probs(level, cost, n_need, contesters, bg_scale)
    if pr is None:
        return np.inf
    pr = pr[owned:]
    if len(pr) == 0:
        return 0.0
    if (pr <= 0).any():
        return np.inf
    Q = transition_Q(pr)
    fundamental = np.linalg.inv(np.eye(len(pr)) - Q)
    shops = fundamental[0].sum()
    return REROLL_COST * max(shops - 1, 0) + cost * (n_need - owned)


def simulate_gold(level, cost, n_need, owned=0, contesters=0, bg_scale=1.0, n_sim=20000, seed=0):
    '''Monte Carlo of the same chain, returns gold spent per simulated run.'''
    pr = slot_probs(level, cost, n_need, contesters, bg_scale)
    if pr is None or (pr[owned:] <= 0).any():
        return np.full(n_sim, np.inf)
    rng = np.random.default_rng(seed)
    p_full = np.append(pr, 0.0)
    s = np.full(n_sim, owned)
    shops = np.zeros(n_sim, dtype=int)
    alive = s < n_need
    while alive.any() and shops.max() < 20000:
        hits = rng.binomial(SLOTS, p_full[s])
        s = np.where(alive, np.minimum(n_need, s + hits), s)
        shops += alive
        alive = s < n_need
    return REROLL_COST * np.maximum(shops - 1, 0) + cost * (n_need - owned)

## 4. Sanity check: does the analytic answer match simulation?
Scenario: level 8, a 3-cost unit, I already hold 1 copy and want a 2-star, one other player is also on the unit.

In [ ]:
sc = dict(level=8, cost=3, n_need=3, owned=1, contesters=1)
sim = simulate_gold(**sc)
print("Analytic expected gold :", round(expected_gold(**sc), 1))
print("Monte Carlo mean       :", round(sim.mean(), 1))
print("Monte Carlo median     :", round(float(np.median(sim)), 1))
print("Monte Carlo 90th pct   :", round(float(np.percentile(sim, 90)), 1), "(bad-luck case)")

## 5. Price list: expected gold to reach a 2-star and a 3-star, from zero copies

In [ ]:
rows = []
for star, n_need in COPIES_FOR_STAR.items():
    for cost in range(1, 6):
        for level in range(1, 12):
            if ODDS.loc[level, cost] == 0:
                continue
            for c in range(4):
                rows.append(dict(star=star, cost=cost, level=level, contesters=c,
                                 gold=expected_gold(level, cost, n_need, 0, c)))
prices = pd.DataFrame(rows)

for star in (2, 3):
    print(f"Expected gold to a {star}-star, uncontested (blank = never feasible or cost not offered):")
    t = (prices[(prices.star == star) & (prices.contesters == 0)]
         .pivot(index="level", columns="cost", values="gold").replace(np.inf, np.nan).round(0))
    display(t)

In [ ]:
finite = prices[np.isfinite(prices["gold"]) & (prices["contesters"] == 0)]
best = (finite.loc[finite.groupby(["star", "cost"])["gold"].idxmin()]
              [["star", "cost", "level", "gold"]].round(0).reset_index(drop=True))
print("Level with the lowest rolling cost per cost tier (uncontested). This ignores the gold needed to REACH that level, so it leans toward high levels:")
best

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, star in zip(axes, [2, 3]):
    for cost in range(1, 6):
        d = prices[(prices.star == star) & (prices.cost == cost) & (prices.contesters == 0) & np.isfinite(prices.gold)]
        if len(d):
            ax.plot(d.level, d.gold, marker="o", label=f"{cost}-cost")
    ax.set_yscale("log")
    ax.set_xlabel("player level")
    ax.set_ylabel("expected gold (log scale)")
    ax.set_title(f"Gold to hit a {star}-star, uncontested")
    ax.legend()
plt.tight_layout()
plt.show()

## 6. What contestation costs
Same price list, at each cost tier's cheapest level, as 0 to 3 other players hold the same unit.
`inf` means the pool cannot supply enough copies at all.

In [ ]:
sens = (prices.merge(best[["star", "cost", "level"]], on=["star", "cost", "level"])
              .pivot_table(index=["star", "cost"], columns="contesters", values="gold").round(0))
sens

## 7. Does the price match what players actually reach?
Observed count of 3-star units in your data by cost tier, next to the modelled price of a 3-star at that tier's cheapest level.
If the model is sensible, expensive-to-hit tiers should be rare on real boards.

In [ ]:
obs = units[units["tier"] >= 3].groupby("cost").size().rename("observed_3star_units")
cmp = best[best["star"] == 3].set_index("cost")[["level", "gold"]].join(obs)
cmp

In [ ]:
prices.to_csv(DATA / "rolling_costs.csv", index=False)
print("saved", DATA / "rolling_costs.csv")

In [ ]:
units[units["cost"] == 5].groupby("unit").size().sort_values()

## Assumptions to state in your write-up
- Hits per shop use a **Binomial(5, p)** approximation. In the real game the 5 slots are drawn without replacement, so the exact law is multivariate hypergeometric. The difference is small when the pool is large and larger for 4- and 5-costs.
- **You buy every copy that appears** and always have the gold and bench space. Real play is gold-limited.
- Background pool depletion is estimated from **final boards**, so it is an upper bound early on and roughly right late.
- Contesters each hold the average number of copies seen on data boards. A real contester's holdings change during the game.
- **No interest and no leveling cost yet.** Spending gold also forfeits interest (up to +5 per round), and levelling costs gold. Both belong to the decision engine in notebook 05.
- Odds and pool sizes come from one site and should be checked against the in-game values for your patch.